# Copilot Product Feedback — Ingester (Fabric)

Lands the **Microsoft 365 Copilot user product feedback** export (the thumbs up/down + verbatim
comments users leave on Copilot responses) into the Lakehouse Delta table **`dbo.user_feedback`**,
which the **ValueLens — 1905 Extra** PBIP reads for the **User Feedback** page.

**Why this exists:** the MAC *Product Feedback* report has **no API** — admins can only *view /
export / delete* it from **Microsoft 365 Admin Center → Health → Product Feedback**. So, exactly like
the Power Platform credit-consumption reports, the CSV is **landed into the Lakehouse by a Power
Automate flow** (`3. Fabric/flows/Copilot_ProductFeedback_Email_to_OneLake.json`) and this notebook
turns it into the table the dashboard expects.

```
M365 Admin Center  ─(export)─▶  Power Automate flow  ─▶  Lakehouse Files/product_feedback/*.csv  ─▶  THIS notebook  ─▶  dbo.user_feedback
```

**Column names are preserved verbatim** (`Feedback Id`, `Logs, Attachments`, `AI Context Prompt`, …)
via **Delta column mapping** — the PBIP's `ProductFeedback` query renames them downstream, so they
must match the export exactly. Two columns are **derived**: `Date Submitted Date` (date part of
`Date Submitted UTC`) and `Sentiment` (Positive / Negative from the thumbs/smiley `Feedback Type`).

**Graceful by design** — feedback is *optional*. No file ⇒ an empty, correctly-named table, and the
PBIP's `Enable_ProductFeedback = "Exclude"` toggle keeps the page dormant. **No app registration /
Graph permission required** — this notebook only reads files already in the Lakehouse.

## 1. Configuration

Tag this cell as the pipeline **`parameters`** cell so a Fabric Pipeline can override `WRITE_MODE`.
`TARGET_TABLE` is schema-qualified (`dbo.`) to work with schema-enabled and legacy Lakehouses.

In [ ]:
# === CONFIG ===  (tag this cell as the pipeline `parameters` cell)

# Folder in the attached Lakehouse where the Power Automate flow lands the feedback CSV(s).
SOURCE_DIR   = 'Files/product_feedback'

# Case-insensitive filename glob. Matches the MAC export whether it's named
# 'feedback.csv', 'ProductFeedback_*.csv', 'extracted_user_feedback.csv', etc.
REPORT_GLOB  = '*feedback*'

TARGET_TABLE = 'dbo.user_feedback'

WRITE_MODE   = 'overwrite'   # current-state snapshots only; append is rejected below
ADD_LINEAGE  = True          # add SourceFile + LoadDate columns to every row
STRICT       = False         # True = raise when no file matches
ALLOW_EMPTY_FIRST_SNAPSHOT = False   # set True only to create an explicit first-install placeholder


## 2. Helpers

`_list_matches` resolves the glob against the Lakehouse `Files/` area (local `/lakehouse/default/`
mount). `_read_csv` parses with multiline + quote-escape so verbatim comments survive. `_enrich`
adds the two derived columns. **Note:** unlike the consumption ingester, column names are **not**
sanitised — the spaced names are kept and Delta column mapping carries them to the SQL endpoint.

In [ ]:
import os, re, datetime
import notebookutils
from pyspark.sql import functions as F

_LOAD_TS = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

def _list_matches(pattern):
    # OneLake metadata works in both notebook and Livy sessions; /lakehouse/default
    # is a notebook-local mount and is not guaranteed in other Spark entry points.
    if not notebookutils.fs.exists(SOURCE_DIR):
        return []
    rx = re.compile('^' + re.escape(pattern).replace('\\*', '.*') + '$', re.IGNORECASE)
    entries = notebookutils.fs.ls(SOURCE_DIR)
    names = sorted(entry.name for entry in entries
                   if not entry.isDir and rx.match(entry.name) and entry.name.lower().endswith('.csv'))
    return [SOURCE_DIR.rstrip('/') + '/' + name for name in names]


def _read_csv(path):
    return (spark.read
            .option('header', True)
            .option('multiLine', True)
            .option('escape', '"')
            .option('encoding', 'UTF-8')
            .csv(path))

# Sentiment derived from the thumbs / smiley Feedback Type; Date from Date Submitted UTC (MM/dd/yyyy HH:mm:ss).
_POS = ['thumbs up', 'smile', 'like', 'compliment', 'positive']
_NEG = ['thumbs down', 'frown', 'dislike', 'complaint', 'negative']

def _enrich(df):
    cols = [c.lstrip('\ufeff') for c in df.columns]
    df = df.toDF(*cols)                      # strip any BOM, keep spaces/case
    if 'Feedback Type' in df.columns:
        ft = F.lower(F.trim(F.col('`Feedback Type`')))
        df = df.withColumn('Sentiment',
                           F.when(ft.isin(_POS), F.lit('Positive'))
                            .when(ft.isin(_NEG), F.lit('Negative'))
                            .otherwise(F.lit(None).cast('string')))
    if 'Date Submitted UTC' in df.columns:
        c = F.col('`Date Submitted UTC`')
        ts = F.coalesce(
            F.to_timestamp(c, 'MM/dd/yyyy HH:mm:ss'),
            F.to_timestamp(c, 'M/d/yyyy H:mm:ss'),
            F.to_timestamp(c, 'yyyy-MM-dd HH:mm:ss'),
            F.to_timestamp(c))
        df = df.withColumn('Date Submitted Date', ts.cast('date'))
    return df

print(f'Load timestamp: {_LOAD_TS}')
print(f'Source folder : {SOURCE_DIR}  (exists: {notebookutils.fs.exists(SOURCE_DIR)})')

## 3. Ingest → `dbo.user_feedback`

Unions every matching CSV, enriches, adds lineage, and writes with **Delta column mapping** so the
spaced/comma column names (`Feedback Id`, `Logs, Attachments`, …) survive to the SQL endpoint. A
missing file writes an **empty, correctly-named** table so the PBIP query never errors.

In [ ]:
# Snapshot validation and the single publication step are in the final cell below.


## 4. Verify

Headline counts + the thumbs/sentiment split, so you can sanity-check the load.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import Window

# The 23-column contract the PBIP 'ProductFeedback' query expects (names kept verbatim).
EMPTY_SCHEMA = ['Feedback Id', 'Comment', 'Translated Comment', 'Comment Language', 'Date Submitted UTC',
    'Feedback Type', 'Microsoft Response Status', 'App', 'App Language', 'Platform', 'Source Type',
    'Logs, Attachments', 'User Id', 'User Email', 'Browser', 'Browser Version', 'AI Context Prompt',
    'AI Context Response Message', 'Survey Question', 'Survey Response Option', 'Additional Metadata',
    'Date Submitted Date', 'Sentiment']


def _table_exists(table_name):
    return bool(spark.catalog.tableExists(table_name))


def _assert_feedback_write_mode(write_mode):
    if str(write_mode).strip().lower() != 'overwrite':
        raise ValueError(
            "WRITE_MODE='append' is not supported for user_feedback snapshots. "
            'Use overwrite, or implement a deterministic merge keyed by Feedback Id.'
        )


def _decide_missing_feedback_action(matches, table_exists, strict, allow_empty_first_snapshot):
    if matches:
        return {'action': 'load', 'message': f'loading {len(matches)} file(s)'}
    msg = f'no file matched "{REPORT_GLOB}" in {SOURCE_DIR}'
    if strict:
        raise FileNotFoundError(msg)
    if table_exists:
        return {'action': 'preserve', 'message': f'⚠  {msg}; Preserving existing {TARGET_TABLE} snapshot.'}
    if not allow_empty_first_snapshot:
        raise ValueError(
            f'{msg}; set ALLOW_EMPTY_FIRST_SNAPSHOT = True to write an explicit empty first-install placeholder.'
        )
    return {'action': 'write-empty', 'message': f'⚠  {msg}; writing explicit empty first-install placeholder.'}


def _empty(cols):
    cols = list(cols) + (['SourceFile', 'LoadDate'] if ADD_LINEAGE else [])
    return spark.createDataFrame([], StructType([StructField(c, StringType(), True) for c in cols]))


def _write(df, table):
    (df.write.mode(WRITE_MODE)
        .option('overwriteSchema', 'true')
        .option('delta.columnMapping.mode', 'name')
        .option('delta.minReaderVersion', '2')
        .option('delta.minWriterVersion', '5')
        .format('delta').saveAsTable(table))


def _feedback_order_expr(df):
    if 'Date Submitted UTC' not in df.columns:
        return F.lit(None).cast('timestamp')
    col = F.col('`Date Submitted UTC`')
    return F.coalesce(
        F.to_timestamp(col, 'MM/dd/yyyy HH:mm:ss'),
        F.to_timestamp(col, 'M/d/yyyy H:mm:ss'),
        F.to_timestamp(col, 'yyyy-MM-dd HH:mm:ss'),
        F.to_timestamp(col),
    )


_assert_feedback_write_mode(WRITE_MODE)
matches = _list_matches(REPORT_GLOB)
decision = _decide_missing_feedback_action(
    matches=matches,
    table_exists=_table_exists(TARGET_TABLE),
    strict=STRICT,
    allow_empty_first_snapshot=ALLOW_EMPTY_FIRST_SNAPSHOT,
)
print(decision['message'])

if decision['action'] == 'preserve':
    rows = spark.table(TARGET_TABLE).count()
elif decision['action'] == 'write-empty':
    _write(_empty(EMPTY_SCHEMA), TARGET_TABLE)
    rows = 0
else:
    frames = []
    for path in matches:
        frame = _enrich(_read_csv(path))
        if ADD_LINEAGE:
            frame = frame.withColumn('SourceFile', F.lit(os.path.basename(path))) \
                         .withColumn('LoadDate', F.lit(_LOAD_TS))
        frames.append(frame)

    df = frames[0]
    for frame in frames[1:]:
        df = df.unionByName(frame, allowMissingColumns=True)

    if 'Feedback Id' not in df.columns:
        raise ValueError("Feedback export is missing 'Feedback Id'; refusing to write an unstable snapshot.")
    blank_ids = df.filter(F.col('`Feedback Id`').isNull() | (F.trim(F.col('`Feedback Id`')) == '')).count()
    if blank_ids:
        raise ValueError(f'Feedback export contains {blank_ids:,} blank Feedback Id value(s); refusing to dedupe an unstable snapshot.')

    for column in EMPTY_SCHEMA:
        if column not in df.columns:
            df = df.withColumn(column, F.lit(None).cast('string'))

    nonblank_score = sum(
        F.when(F.col(f'`{column}`').isNotNull() & (F.trim(F.col(f'`{column}`').cast('string')) != ''), F.lit(1)).otherwise(F.lit(0))
        for column in df.columns if column != 'Feedback Id'
    )
    order_exprs = [_feedback_order_expr(df).desc_nulls_last(), nonblank_score.desc()]
    if ADD_LINEAGE and 'SourceFile' in df.columns:
        order_exprs.append(F.col('SourceFile').desc_nulls_last())
    window = Window.partitionBy(F.col('`Feedback Id`')).orderBy(*order_exprs)
    df = (df.withColumn('_feedback_nonblank', nonblank_score)
            .withColumn('_feedback_row', F.row_number().over(window))
            .filter(F.col('_feedback_row') == 1)
            .drop('_feedback_nonblank', '_feedback_row'))

    rows = df.count()
    if rows == 0 and _table_exists(TARGET_TABLE):
        raise ValueError(f'Feedback export parsed 0 rows; refusing to replace existing {TARGET_TABLE}.')
    if rows == 0 and not ALLOW_EMPTY_FIRST_SNAPSHOT:
        raise ValueError(
            'Feedback export parsed 0 rows. Set ALLOW_EMPTY_FIRST_SNAPSHOT = True only '
            'for an intentional empty first-install placeholder.'
        )

    _write(df, TARGET_TABLE)
    print(f'✓  {TARGET_TABLE}: {len(matches)} file(s), {rows:,} rows -> written ({WRITE_MODE})')

print('Done. Refresh the PBIP / Direct Lake model to pick up user_feedback.')
